In [1]:
# import required packages
from sklearn.cluster import DBSCAN
from sklearn.cluster import HDBSCAN
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import folium
import os
import yaml
from pprint import pprint
import pandas as pd
import geopandas as gpd
from scipy.spatial import ConvexHull
import numpy as np

from pyproj import Transformer

sns.set_theme(context='paper', style='darkgrid')

In [2]:
INDO_CRS = "EPSG:23867"
LL_CRS = "EPSG:4326"

def xy_to_ll(xy):
        return Transformer.from_crs(INDO_CRS, LL_CRS).transform(xy[0], xy[1])

def ll_to_xy(ll):
    return Transformer.from_crs(LL_CRS, INDO_CRS).transform(ll[0], ll[1])

In [3]:
# construct dataset for intermediaries
dfs = []
root = '../FactoredPlatformSolver/data/instances'

for f in os.listdir(root):
    if f.endswith('.yaml') and not f.startswith('aggregate'):
        with open(os.path.join(root, f), 'r') as file:
            data = yaml.safe_load(file)
            df_i = pd.DataFrame(data['intermediaries']).drop(['capacity', 'routes'], axis=1)
            dfs.append(df_i)

ints_df = pd.concat(dfs)

ints_df['int_lat'] = ints_df.location.apply(lambda x: x[0])
ints_df['int_lon'] = ints_df.location.apply(lambda x: x[1])

ints_df['int_x'], ints_df['int_y'] = ll_to_xy([ints_df.int_lat.values, ints_df.int_lon.values])

ints_df.drop('location', axis=1, inplace=True)

ints_df.rename(
    columns={'id': 'int_id'},
    inplace=True
)

ints_df.drop_duplicates(inplace=True)

ints_df.reset_index(drop=True, inplace=True)

ints_df.to_csv('ints_2.csv', index=False)

ints_df


,int_id,int_lat,int_lon,int_x,int_y
0,Dodi Lesmana,-0.39393,102.45772,884983.940054,-43621.049913
1,Purnomo,-0.68244,102.54698,894916.216941,-75575.896083
2,Isna,-0.70056,102.50660,890413.186074,-77579.179968
3,Agus Wibowo,-0.68910,102.46092,885321.983167,-76306.384048
4,Nurmala,-0.53178,102.40239,878808.787091,-58882.167437
5,yaya suhayat,-0.41070,102.47983,887447.924499,-45479.113640
6,Agus Yasir,-0.51696,102.39634,878135.299269,-57240.837871
7,Ngatinu,-0.52602,102.40703,879326.350243,-58244.664749
8,Samsuri,-0.35302,102.44236,883273.449125,-39090.326151
9,Riki Mandala,-0.39021,102.45724,884930.601578,-43209.101061


In [5]:
# construct master dataset for mills
dfs = []

for f in os.listdir(root):
    if f.endswith('.yaml') and not f.startswith('aggregate'):
        with open(os.path.join(root, f), 'r') as file:
            data = yaml.safe_load(file)
            df_m = pd.DataFrame(data['mills'])
            dfs.append(df_m)

mills_df = pd.concat(dfs)

mills_df['mill_lat'] = mills_df.location.apply(lambda x: x[0])
mills_df['mill_lon'] = mills_df.location.apply(lambda x: x[1])

mills_df['mill_x'], mills_df['mill_y'] = ll_to_xy([mills_df.mill_lat.values, mills_df.mill_lon.values])

mills_df.drop('location', axis=1, inplace=True)

mills_df.rename(
    columns={'id': 'mill_id'},
    inplace=True
)

mills_df.drop_duplicates(inplace=True)

mills_df.reset_index(drop=True, inplace=True)

mills_df.to_csv('mills_2.csv', index=False)

mills_df

,mill_id,mill_lat,mill_lon,mill_x,mill_y
0,NHR,-0.733834,102.524376,892391.975126,-81265.464763
1,Inecda,-0.492910,102.367313,874901.160821,-54576.235330
2,PERSADA AGRO SAWITA,-0.411250,102.359500,874034.479367,-45534.265071
3,SRJ,-0.793145,102.596242,900398.271564,-87840.525932
4,BUMI PALMA,-0.598056,102.983333,943581.055886,-66264.324536
5,PN,-0.374579,102.384353,876806.399799,-41475.058961
6,SKIP,-0.682643,102.501522,889848.574101,-75594.658547


In [6]:
# construct master dataset for all instances

dfs = []

for f in os.listdir(root):
    if f.endswith('.yaml') and not f.startswith('aggregate'):
        with open(os.path.join(root, f), 'r') as file:
            data = yaml.safe_load(file)
        date = f.split('.')[0]

        farmers = pd.DataFrame(data['farmers']).set_index('id')

        ints = data['intermediaries']


        pickup_data = []

        for i in ints:

            if len(i['routes']) == 0:
                continue

            route = i['routes'][0]

            f_data = farmers.loc[route].reset_index()

            pickup_data.append(f_data)

        daily_df = pd.concat(pickup_data)

        daily_df['date'] = pd.to_datetime(date)
        dfs.append(daily_df)
        
farmers_df = pd.concat(dfs)

farmers_df.rename(
    columns={'id': 'farmer_id',
            'intermediary': 'int_id',
            'location': 'farmer_loc'},
    inplace=True
)

farmers_df = farmers_df.merge(ints_df, on="int_id", how="left")

farmers_df['farmer_lat'] = farmers_df['farmer_loc'].apply(lambda loc: loc[0])
farmers_df['farmer_lon'] = farmers_df['farmer_loc'].apply(lambda loc: loc[1])
farmers_df.drop('farmer_loc', axis=1, inplace=True)

farmers_df['farmer_x'], farmers_df['farmer_y'] = ll_to_xy([farmers_df.farmer_lat.values, farmers_df.farmer_lon.values])

farmers_df['offset_x'] = farmers_df['farmer_x'] - farmers_df['int_x']
farmers_df['offset_y'] = farmers_df['farmer_y'] - farmers_df['int_y']

farmers_df['distance'] = (farmers_df['offset_x'] ** 2 + farmers_df['offset_y'] ** 2) ** 0.5
farmers_df.drop(['offset_x', 'offset_y'], axis=1, inplace=True)

farmers_df.to_csv('farmers_2.csv', index=False)

farmers_df

,farmer_id,int_id,quantity,date,int_lat,int_lon,int_x,int_y,farmer_lat,farmer_lon,farmer_x,farmer_y,distance
0,Dodi Lesmana_18_1,Dodi Lesmana,0.7,2020-09-02,-0.39393,102.45772,884983.940054,-43621.049913,-0.411345,102.45536,884720.037256,-45549.351401,1946.276269
1,Dodi Lesmana_19_5,Dodi Lesmana,1.3,2020-09-02,-0.39393,102.45772,884983.940054,-43621.049913,-0.366220,102.45672,884873.693000,-40552.594938,3070.434879
2,Purnomo_13_11,Purnomo,0.9,2020-09-02,-0.68244,102.54698,894916.216941,-75575.896083,-0.712260,102.52794,892791.142584,-78876.642584,3925.667903
3,Purnomo_7_9,Purnomo,1.3,2020-09-02,-0.68244,102.54698,894916.216941,-75575.896083,-0.696285,102.56225,896617.413683,-77110.427937,2291.038710
4,Purnomo_20_7,Purnomo,1.0,2020-09-02,-0.68244,102.54698,894916.216941,-75575.896083,-0.703750,102.56461,896879.886086,-77937.344821,3071.227223
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,Riki Mandala_58_9,Riki Mandala,1.5,2020-09-01,-0.39021,102.45724,884930.601578,-43209.101061,-0.359335,102.47666,887096.858728,-39791.043483,4046.700835
301,Riki Mandala_25_11,Riki Mandala,1.4,2020-09-01,-0.39021,102.45724,884930.601578,-43209.101061,-0.333080,102.46555,885859.394643,-36883.252882,6393.669662
302,Syafrial_24_4,Syafrial,3.0,2020-09-01,-0.39437,102.48981,888561.265951,-43671.267163,-0.379210,102.45915,885144.016536,-41991.123636,3807.949032
303,Syafrial_8_3,Syafrial,0.5,2020-09-01,-0.39437,102.48981,888561.265951,-43671.267163,-0.407395,102.50487,890239.556892,-45114.343725,2213.397942


In [7]:
f_df = pd.read_csv('data/farmers.csv')
f2_df = pd.read_csv('data/farmers_2.csv')

In [12]:
len(f_df)

2472

In [15]:
print(len(f_df.drop_duplicates(['farmer_lat', 'farmer_lon'])))
print(len(f2_df.drop_duplicates(['farmer_lat', 'farmer_lon'])))

554
233
